In [ ]:
import warnings

import astropy.units
import FunctionLib as FL
import inspect
from tqdm import tqdm
import astropy
import wave
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import defaultdict
import re
import scipy

mpl.rcParams['font.family'] = 'serif'


warnings.filterwarnings("ignore")

DJAv4Catalog = FL.Spectrum_Catalog()
DJAv4Catalog.load_from_pkl(os.path.expanduser(
    './DJAV4.2Catalog.pkl'))
print(DJAv4Catalog.sample_num())

DJAv4Catalog.to_dataframe()

In [ ]:
from importlib import resources
from pathlib import Path
from urllib import request

import matplotlib.pyplot as plt
import numpy as np

from ppxf.ppxf import ppxf
import ppxf.ppxf_util as util
import ppxf.sps_util as lib

import astropy.units as u
import astropy.constants as const
from astropy.io import fits as asfits

In [ ]:

for surveyid_subid, catalog in DJAv4Catalog.catalog_iterator():
    ppxf=False
    if not catalog.get('sample_flag', False):
        continue
    for disperser_filter, spectrum_filepath in catalog['grating_within_coverage'].items():
        spectrum_=FL.Load_Spectrum_From_Fits(spectrum_filepath,catalog['determined_redshift'])
        dual_bounds = spectrum_.dual_boundarys()

        if 'm' in disperser_filter:
            R=1000
        elif 'h' in disperser_filter:
            R=2700

        spectrum_.set_boundarys(3500*u.AA, 6000*u.AA)


        lam=spectrum_.processing_wavelengths.convert_unit_to(u.AA).data
        galaxy=spectrum_.processing_flux_lambda.data
        #if all np.isnan(galaxy):


        FWHM_gal = (1e4*np.sqrt(dual_bounds[0]*dual_bounds[1])/R).value
        print( f"FWHM_gal: {FWHM_gal:.1f} Å")   # 8.5 Angstrom

        c = 299792.458                      # speed of light in km/s
        sigma_inst = c/(R*2.355)
        print( f"sigma_inst: {sigma_inst:.0f} km/s")   # 47 km/s


        z = catalog['determined_redshift']                     # Initial estimate of the galaxy redshift
        # lam /= (1 + z)               # Compute approximate restframe wavelength
        FWHM_gal /= (1 + z)     # Adjust resolution in Angstrom
        print(f"de-redshifted NIRSpec G235H/F170LP resolution FWHM in Å: {FWHM_gal:.1f}")


        galaxy = galaxy/np.nanmedian(galaxy)       # Normalize spectrum to avoid numerical issues
        noise = np.full_like(galaxy, 0.05)      # Assume constant noise per pixel here. I adopt a noise that gives chi2/DOF~1

        sps_name = 'fsps'

        ppxf_dir = resources.files("ppxf")
        basename = f"spectra_{sps_name}_9.0.npz"
        filename = ppxf_dir / 'sps_models' / basename
        if not filename.is_file():
            url = "https://raw.githubusercontent.com/micappe/ppxf_data/main/" + basename
            request.urlretrieve(url, filename)


        d_ln_lam = np.log(lam[-1]/lam[0])/(lam.size - 1)  # Average ln_lam step
        velscale = c*d_ln_lam                   # eq. (8) of Cappellari (2017)
        print(f"Velocity scale per pixel: {velscale:.2f} km/s")


        FWHM_temp = 2.51   # Resolution of E-MILES templates in the fitted range


        sps = lib.sps_lib(filename, velscale, norm_range=[5070, 5950], age_range=[0, 2.2])


        reg_dim = sps.templates.shape[1:]
        stars_templates = sps.templates.reshape(sps.templates.shape[0], -1)

        lam_range_gal = [np.min(lam), np.max(lam)]
        gas_templates, gas_names, line_wave = util.emission_lines(sps.ln_lam_temp, lam_range_gal, FWHM_gal, tie_balmer=1)

        templates = np.column_stack([stars_templates, gas_templates])

        c = 299792.458
        start = [1200, 200.]

        n_stars = stars_templates.shape[1]
        n_gas = len(gas_names)
        component = [0]*n_stars + [1]*n_gas
        gas_component = np.array(component) > 0  # gas_component=True for gas templates

        moments = [2, 2]
        start = [start, start]

        if not np.isfinite(galaxy).any():
            print("All NaN or Inf spectrum, skip pPXF fitting.")
            continue

        try:

            pp = ppxf(templates, galaxy, noise, velscale, start,
            moments=moments, degree=-1, mdegree=-1, lam=lam, lam_temp=sps.lam_temp,
            reg_dim=reg_dim, component=component, gas_component=gas_component,
            reddening=0, gas_reddening=0, gas_names=gas_names)
            plt.figure(figsize=(15, 5))
            pp.plot()
            plt.title(f"pPXF fit with {sps_name} SPS templates")


            plt.figure(figsize=(15, 5))
            pp.plot(gas_clip=1)
            plt.title(f"pPXF fit with {sps_name} SPS templates")
            plt.xlim([0.42, 0.52])

            ppxf=True
            break
        except Exception as e:
            print(f"pPXF fitting failed: {e}")
            break
    break

FWHM_gal: 6.2 Å
sigma_inst: 127 km/s
de-redshifted NIRSpec G235H/F170LP resolution FWHM in Å: 1.5
Velocity scale per pixel: 152.09 km/s
Emission lines included in gas templates:
['Balmer' '[NeIII]3968' '[NeIII]3869' 'HeII4687' 'HeI5876' '[OIII]5007_d']
pPXF fitting failed: 'bool' object is not callable


In [21]:
gas_names

array(['Balmer', 'HeII4687', 'HeI5876', '[OIII]5007_d'], dtype='<U12')